# One-Phase Shock Tube Validation – Colocated vs VDF

In [ ]:
from trustutils import run
run.introduction('W. Aboussi, K. Pons and E. Saikali')

### Description des cas tests

These tests are designed to assess the robustness and accuracy of numerical methods at the core of the solver, independently from the boundary condition, and source term treatment involving correla       tions.

They consist in the numerical resolution of the Shock tube problem for a perfect gas with $\gamma=1.4$. 

The initial state consists in two constant states $W_L=(\rho_L,u_L,p_L)$ and $W_R=(\rho_R,u_R,p_R)$ separated by a discontinuity at $x=x_d$. The following table gives the values of $W_L$ and $W_R$ for each test.

In [ ]:
from trustutils import run
run.TRUST_parameters()

In [ ]:
TCs = {
    "Toro1": {"WL": [1, 0.75, 1], "WR": [0.125, 0, 0.1], "tmax": 0.2, "gamma": 1.4, "pinf": 0, "x": 0.3},
    "Toro2": {"WL": [1, -2, 0.4], "WR": [1, 2, 0.4], "tmax": 0.15, "gamma": 1.4, "pinf": 0, "x": 0.5},
    "Toro3": {"WL": [1, 0, 1000], "WR": [1, 0, 0.01], "tmax": 0.012, "gamma": 1.4, "pinf": 0, "x": 0.5},
    "Toro4": {
        "WL": [5.99924, 19.5975, 460.894],
        "WR": [5.99242, -6.19633, 46.0950],
        "tmax": 0.035,
        "gamma": 1.4,
        "pinf": 0,
        "x": 0.4,
    },
    "Toro5": {
        "WL": [1, -19.59745, 1000],
        "WR": [1, -19.59745, 0.01],
        "tmax": 0.012,
        "gamma": 1.4,
        "pinf": 0,
        "x": 0.8,
    },
    "Toro6": {"WL": [1.4, 0, 1], "WR": [1.4, 0, 1], "tmax": 2, "gamma": 1.4, "pinf": 0, "x": 0.5},
    "Toro7": {"WL": [1.4, 0.1, 1], "WR": [1, 0.1, 1], "tmax": 2, "gamma": 1.4, "pinf": 0, "x": 0.5},
    "PWR1": {
        "WL": [700, 0, 155e5],
        "WR": [700, 0, 1e5],
        "tmax": 3e-4,
        "gamma": 1.58,
        "pinf": 353637173.0,
        "x": 0.5,
    },
    "PWR2": {
        "WL": [700, 0, 155e5],
        "WR": [700, 20, 155e5],
        "tmax": 3e-4,
        "gamma": 1.58,
        "pinf": 353637173.0,
        "x": 0.5,
    },
    "PWR3": {
        "WL": [700, 0, 155e5],
        "WR": [650, 0, 155e5],
        "tmax": 3e-4,
        "gamma": 1.58,
        "pinf": 353637173.0,
        "x": 0.5,
    },
}

In [ ]:
from trustutils import plot
from IPython.display import display

#tests = ['Toro1','Toro2','Toro4','Toro5','Toro6','Toro7','PWR1','PWR2','PWR3']
tests = ['Toro1','Toro2','Toro4','Toro5']
columns = [r"$\gamma$", r"$p_\infty$", r"$\rho_L$", r"$u_L$", r"$p_L$", r"$\rho_R$", r"$u_R$", r"$p_R$",r"$x_d$"]
tab = plot.Table(columns)
for name in tests:
    tab.addLine( [[TCs[name]["gamma"], TCs[name]["pinf"]] + TCs[name]["WL"] + TCs[name]["WR"] + [TCs[name]["x"]]], name)

meshes =[200]
run.reset()

display(tab)
for m in meshes :
    dic={}
    dic["__n__"] = str(m + 1)
    dic["__dis__"] = "Coloc"
    for name in tests:
        dt = TCs[name]["tmax"] / m
        dic["__dt__"] = str(dt)
        for i, n in enumerate(["__rl__", "__vl__", "__pl__"]):
            dic[n] = str(TCs[name]["WL"][i])
        for i, n in enumerate(["__rr__", "__vr__", "__pr__"]):
            dic[n] = str(TCs[name]["WR"][i])
        for n in ["tmax", "gamma", "pinf", "x"]:
            dic["__{}__".format(n)] = str(TCs[name][n])
        run.addCaseFromTemplate("jdd.data",targetDirectory=f"n{m}/Coloc", targetData=f"{name}.data",dic=dic)
        
run.printCases()
run.runCases()
run.tablePerf()

In [ ]:
def plot_baltik(list_meshes,list_tests):

    from trustutils import plot
    variables = ["RHO", "P", "V"]
    dis_m = {"Coloc" : "o", "VDF" : "x"}
    for test in list_tests:
        df = plot.read_csv(f"exact/{test}.ex", sep=r'\s+', usecols=[0,1,2,3,4])
        a = plot.Graph(nX=1, nY=3, title=test)
        nX, nY = 0, 0
        for p in variables:
         a.addPlot(nY, f"Profil de {p}")
         for m in list_meshes:
            for dis, ma in dis_m.items():
                if dis == "Coloc":
                    a.addSegment(f"n{m}/{dis}/{test}_{p}.son", label=f"{dis} n={m}",lw = 2, marker=ma)
                else :
                     path_trust_results="vdf"
                     a.addSegment(f"{path_trust_results}/n{m}/{dis}/{test}_{p}.son", label=f"{dis} n={m}",lw = 2, marker=ma)
         a.add(df["x"], df[f"{p}"], label="Exact solution", lw=1, color='k', marker="--")
         a.label("z [m]", p)
         
         nY = nY+1 

plot_baltik(meshes,tests)